# 📊 Bluestock Fintech — Mutual Fund Analytics Platform
## Day 1 | EDA Notebook — Fund Master Exploration & AMFI Code Validation

**Tasks covered:**
- Task 6 : Understand fund master — unique fund houses, categories, sub-categories, risk grades
- Task 7 : Validate AMFI codes — check all codes in fund_master exist in nav_history

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# ── Style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#3a3d4d',
    'axes.labelcolor':  '#c8cad4',
    'text.color':       '#c8cad4',
    'xtick.color':      '#c8cad4',
    'ytick.color':      '#c8cad4',
    'grid.color':       '#2a2d3d',
    'grid.alpha':       0.5,
    'figure.dpi':       120,
    'font.family':      'DejaVu Sans',
})
ACCENT = ['#7B61FF', '#00D4AA', '#FF6B6B', '#FFB347', '#4ECDC4', '#A8DADC',
          '#E63946', '#457B9D', '#F4A261', '#2A9D8F']

# ── Paths ──────────────────────────────────────────────────────────────────
RAW_DIR = Path('data/raw')
REPORT_DIR = Path('reports')
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded ✔')

---
## 1. Load All 10 Datasets

In [ ]:
fund_master  = pd.read_csv(RAW_DIR / '01_fund_master.csv', parse_dates=['launch_date'])
nav_history  = pd.read_csv(RAW_DIR / '02_nav_history.csv', parse_dates=['date'])
aum_fh       = pd.read_csv(RAW_DIR / '03_aum_by_fund_house.csv', parse_dates=['date'])
sip_inflows  = pd.read_csv(RAW_DIR / '04_monthly_sip_inflows.csv')
cat_inflows  = pd.read_csv(RAW_DIR / '05_category_inflows.csv')
folio_count  = pd.read_csv(RAW_DIR / '06_industry_folio_count.csv')
scheme_perf  = pd.read_csv(RAW_DIR / '07_scheme_performance.csv')
investor_tx  = pd.read_csv(RAW_DIR / '08_investor_transactions.csv')
portfolio    = pd.read_csv(RAW_DIR / '09_portfolio_holdings.csv', parse_dates=['portfolio_date'])
benchmark    = pd.read_csv(RAW_DIR / '10_benchmark_indices.csv', parse_dates=['date'])

datasets = {
    '01_fund_master':          fund_master,
    '02_nav_history':          nav_history,
    '03_aum_by_fund_house':    aum_fh,
    '04_monthly_sip_inflows':  sip_inflows,
    '05_category_inflows':     cat_inflows,
    '06_industry_folio_count': folio_count,
    '07_scheme_performance':   scheme_perf,
    '08_investor_transactions':investor_tx,
    '09_portfolio_holdings':   portfolio,
    '10_benchmark_indices':    benchmark,
}

print('{:<35} {:>8}  {:>4}'.format('Dataset', 'Rows', 'Cols'))
print('-' * 52)
for name, df in datasets.items():
    print(f'{name:<35} {len(df):>8,}  {df.shape[1]:>4}')

In [ ]:
print('=== DATASET INSPECTION ===')
for name, df in datasets.items():
    print('\n' + '=' * 96)
    print(name)
    print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
    print('Dtypes:')
    print(df.dtypes.to_string())
    print('Head:')
    print(df.head().to_string(index=False))

---
## 2. Task 6 — Fund Master Exploration
### 2.1 Unique Fund Houses

In [ ]:
print('=== UNIQUE FUND HOUSES ===')
fh_counts = fund_master['fund_house'].value_counts()
print(fh_counts.to_string())
print(f'\nTotal fund houses : {fund_master["fund_house"].nunique()}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(fh_counts.index[::-1], fh_counts.values[::-1],
               color=ACCENT[:len(fh_counts)], edgecolor='none', height=0.6)
ax.bar_label(bars, padding=4, color='#c8cad4', fontsize=9)
ax.set_xlabel('Number of Schemes')
ax.set_title('Schemes per Fund House', fontsize=14, fontweight='bold', color='#e0e2f0', pad=12)
ax.grid(axis='x', linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('reports/fund_house_schemes.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()

### 2.2 Unique Categories & Sub-categories

In [ ]:
print('=== UNIQUE CATEGORIES ===')
print(fund_master['category'].value_counts().to_string())
print(f'\nTotal categories : {fund_master["category"].nunique()}')

print('\n=== UNIQUE SUB-CATEGORIES ===')
print(fund_master['sub_category'].value_counts().to_string())
print(f'\nTotal sub-categories : {fund_master["sub_category"].nunique()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Category pie
cat_counts = fund_master['category'].value_counts()
axes[0].pie(cat_counts, labels=cat_counts.index, autopct='%1.0f%%',
            colors=ACCENT[:len(cat_counts)], startangle=140,
            wedgeprops={'edgecolor': '#0f1117', 'linewidth': 2})
axes[0].set_title('Category Distribution', fontsize=13, fontweight='bold',
                   color='#e0e2f0', pad=10)

# Sub-category bar
sub_counts = fund_master['sub_category'].value_counts()
axes[1].barh(sub_counts.index[::-1], sub_counts.values[::-1],
             color=ACCENT[2], edgecolor='none', height=0.6)
axes[1].set_xlabel('Number of Schemes')
axes[1].set_title('Sub-Category Distribution', fontsize=13, fontweight='bold',
                   color='#e0e2f0', pad=10)
axes[1].grid(axis='x', linestyle='--', alpha=0.4)

fig.patch.set_facecolor('#0f1117')
plt.tight_layout()
plt.savefig('reports/category_distribution.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()

### 2.3 Risk Grades

In [ ]:
print('=== UNIQUE RISK GRADES ===')
risk_order = ['Low', 'Moderate', 'Moderately High', 'High', 'Very High']
risk_counts = fund_master['risk_category'].value_counts().reindex(risk_order).dropna()
print(risk_counts.to_string())

In [ ]:
risk_colors = ['#00D4AA', '#7B61FF', '#FFB347', '#FF6B6B', '#E63946']
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(risk_counts.index, risk_counts.values,
              color=risk_colors[:len(risk_counts)], edgecolor='none', width=0.55)
ax.bar_label(bars, padding=4, color='#c8cad4', fontsize=11, fontweight='bold')
ax.set_xlabel('Risk Category')
ax.set_ylabel('Number of Schemes')
ax.set_title('Schemes by Risk Grade (SEBI Riskometer)', fontsize=13,
              fontweight='bold', color='#e0e2f0', pad=12)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
fig.patch.set_facecolor('#0f1117')
plt.tight_layout()
plt.savefig('reports/risk_grade_distribution.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()

### 2.4 Expense Ratio & Plan Type Analysis

In [ ]:
print('=== EXPENSE RATIO STATS ===')
print(fund_master.groupby('plan')['expense_ratio_pct'].describe().round(4).to_string())

print('\n=== PLAN TYPES ===')
print(fund_master['plan'].value_counts().to_string())

print('\n=== CATEGORY × PLAN CROSSTAB ===')
print(pd.crosstab(fund_master['category'], fund_master['plan']).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Expense ratio by plan
for i, (plan, grp) in enumerate(fund_master.groupby('plan')):
    axes[0].hist(grp['expense_ratio_pct'], bins=10, alpha=0.8,
                 label=plan, color=ACCENT[i], edgecolor='#0f1117')
axes[0].set_xlabel('Expense Ratio (%)')
axes[0].set_ylabel('Number of Schemes')
axes[0].set_title('Expense Ratio Distribution by Plan', fontsize=12,
                   fontweight='bold', color='#e0e2f0')
axes[0].legend()
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# Expense ratio by sub-category (equity only)
eq = fund_master[fund_master['category'] == 'Equity']
sub_exp = eq.groupby('sub_category')['expense_ratio_pct'].mean().sort_values(ascending=False)
axes[1].barh(sub_exp.index[::-1], sub_exp.values[::-1],
             color=ACCENT[3], edgecolor='none', height=0.6)
axes[1].set_xlabel('Avg Expense Ratio (%)')
axes[1].set_title('Avg Expense Ratio by Sub-Category (Equity)', fontsize=12,
                   fontweight='bold', color='#e0e2f0')
axes[1].grid(axis='x', linestyle='--', alpha=0.4)

fig.patch.set_facecolor('#0f1117')
plt.tight_layout()
plt.savefig('reports/expense_ratio_analysis.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()

---
## 3. Task 7 — AMFI Code Validation

In [ ]:
master_codes = set(fund_master['amfi_code'].unique())
nav_codes    = set(nav_history['amfi_code'].unique())

missing_in_nav = master_codes - nav_codes
extra_in_nav   = nav_codes - master_codes
matched        = master_codes & nav_codes

print('=== AMFI CODE VALIDATION SUMMARY ===')
print(f'  Codes in fund_master        : {len(master_codes)}')
print(f'  Codes in nav_history        : {len(nav_codes)}')
print(f'  Matched (both)              : {len(matched)}')
print(f'  Missing from nav_history    : {len(missing_in_nav)}')
print(f'  Extra in nav (not in master): {len(extra_in_nav)}')

if missing_in_nav:
    print('\n  ⚠ Missing codes:')
    print(fund_master[fund_master['amfi_code'].isin(missing_in_nav)]
          [['amfi_code','scheme_name','fund_house']].to_string(index=False))
else:
    print('\n  ✔ All fund_master AMFI codes exist in nav_history!')

In [ ]:
# NAV records per scheme
nav_counts = nav_history.groupby('amfi_code').agg(
    records   = ('nav', 'count'),
    first_date= ('date', 'min'),
    last_date = ('date', 'max'),
    null_nav  = ('nav', lambda x: x.isna().sum())
).reset_index()

nav_merged = fund_master[['amfi_code','scheme_name','fund_house','category']].merge(
    nav_counts, on='amfi_code', how='left'
)

print('=== NAV RECORD COUNTS PER SCHEME ===')
print(nav_merged[['amfi_code','scheme_name','records','first_date','last_date','null_nav']]
      .to_string(index=False))

In [ ]:
# Visualise NAV coverage
fig, ax = plt.subplots(figsize=(12, 9))
sorted_df = nav_merged.sort_values('records', ascending=True)
colors = [ACCENT[0] if r >= 1000 else ACCENT[2] for r in sorted_df['records']]
bars = ax.barh(range(len(sorted_df)), sorted_df['records'],
               color=colors, edgecolor='none', height=0.7)

ax.set_yticks(range(len(sorted_df)))
ax.set_yticklabels(
    [f"{row.amfi_code} — {row.scheme_name[:40]}" for _, row in sorted_df.iterrows()],
    fontsize=7.5
)
ax.set_xlabel('Number of NAV Records')
ax.set_title('NAV Record Coverage per Scheme', fontsize=13,
              fontweight='bold', color='#e0e2f0', pad=12)
ax.axvline(nav_merged['records'].mean(), color=ACCENT[1], linestyle='--',
           linewidth=1.5, label=f'Avg: {nav_merged["records"].mean():.0f}')
ax.legend(fontsize=9)
ax.grid(axis='x', linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
fig.patch.set_facecolor('#0f1117')
plt.tight_layout()
plt.savefig('reports/nav_coverage_per_scheme.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()

In [ ]:
# Null/Duplicate check
total_rows  = len(nav_history)
null_rows   = nav_history['nav'].isna().sum()
dup_rows    = nav_history.duplicated(subset=['amfi_code','date']).sum()

print('=== NAV HISTORY DATA QUALITY ===')
print(f'  Total rows             : {total_rows:,}')
print(f'  Null NAV values        : {null_rows}  ({null_rows/total_rows*100:.2f}%)')
print(f'  Duplicate (code+date)  : {dup_rows}')

verdict = 'PASS ✔' if (null_rows + dup_rows) == 0 else 'REVIEW NEEDED ⚠'
print(f'\n  Overall Quality        : {verdict}')

---
## 4. Quick NAV History Snapshot

In [ ]:
# Plot NAV for Large Cap funds
large_cap_codes = fund_master[
    (fund_master['sub_category'] == 'Large Cap') & (fund_master['plan'] == 'Regular')
]['amfi_code'].tolist()

nav_lc = nav_history[nav_history['amfi_code'].isin(large_cap_codes)].copy()
nav_lc = nav_lc.merge(fund_master[['amfi_code','scheme_name']], on='amfi_code', how='left')

# Normalise to 100 at start date
nav_lc = nav_lc.sort_values('date')
start_navs = nav_lc.groupby('amfi_code')['nav'].transform('first')
nav_lc['nav_indexed'] = nav_lc['nav'] / start_navs * 100

fig, ax = plt.subplots(figsize=(13, 6))
for i, (code, grp) in enumerate(nav_lc.groupby('amfi_code')):
    name = grp['scheme_name'].iloc[0].replace(' - Regular Plan - Growth', '').replace(' - Regular - Growth', '')
    ax.plot(grp['date'], grp['nav_indexed'], label=name,
            color=ACCENT[i % len(ACCENT)], linewidth=1.5, alpha=0.9)

ax.axhline(100, color='#555', linestyle='--', linewidth=0.8)
ax.set_xlabel('Date')
ax.set_ylabel('Indexed NAV (Base=100)')
ax.set_title('Large Cap Funds — Indexed NAV (Jan 2022 – May 2026)',
              fontsize=13, fontweight='bold', color='#e0e2f0', pad=12)
ax.legend(fontsize=7.5, loc='upper left', framealpha=0.3)
ax.grid(linestyle='--', alpha=0.3)
fig.patch.set_facecolor('#0f1117')
plt.tight_layout()
plt.savefig('reports/large_cap_nav_indexed.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()

---
## 5. Summary — Day 1 Checkpoint

| Check | Status |
|---|---|
| All 10 CSV files loaded | ✔ |
| fund_master: 40 schemes, 10 fund houses | ✔ |
| nav_history: ~46K rows, 40 schemes | ✔ |
| AMFI code validation | ✔ |
| Null check on nav | ✔ |
| Duplicate check on (code, date) | ✔ |
| Charts saved to reports/ | ✔ |